# SLR2

This notebook covers the screening stage of the review: it checks the title/abstract screening done in ASReview, validates agreement between screeners, and assembles the final set of included studies from the database search and from other methods.

## Notebook structure

- **Imports** — loads all required packages and sets the project root (`base`).
- **1.1 Post Screening - Sample** — summarises the initial calibration screening of 50 records (`asreview_results_initialscreening.csv`).
- **1.2 Post Screening - Full** — loads the full ASReview screening results, checks them against the pre-screened records from SLR1 (`df_final_prescreen.csv`), and draws stratified validation samples of 50 records per student screener.
  - **Compare Screeners** — computes Cohen's kappa between the two student screeners and between each student screener and the lead reviewer.
  - **Fixing Student Screener A's Labels** — exports Student Screener A's remaining records for re-screening, following low agreement in the validation.
  - **Screeners Contribution** — summarises the number of records each screener handled after the re-screening.
- **2. Studies from Databases** — works with the final ASReview export (`asreview_results_FINAL.csv`):
  - **Load in ASReview Output** — extracts the screened and relevant records.
  - **Validate Coverage** — checks that all key papers in the reference set were found and screened as relevant.
  - **Clean & Screen** — supplements metadata from Zotero, updates the publication status of preprints that have since been published, and excludes remaining preprints.
- **3. Studies from Other Methods** — loads records identified outside the database searches, removes preprints and duplicates of database records.
- **4. Merge Identified Records** — combines both sources and cross-references them with the final data extraction table (`DataExtraction_Final.csv`), which is the table is used for Elicit and human coded columns.

## Requirements

The notebook expects the screening files in `data/screening/`, the reference set and other-methods records in `data/search/`, and `DataExtraction_Final.csv` in `data/extraction/` (all relative to the project root).

**Note:** the cells that draw the validation samples and export Student Screener A's records write to `data/screening/` (`df_asreview_sample_*.csv`, `df_asreview_screenerA.csv`) and overwrite the versions shipped with the repository.

# Imports

In [1]:
# Standard Packages 
import numpy as np
import pandas as pd
import unicodedata
import time
import re
import os
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.path import Path
from matplotlib.patches import PathPatch
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime
from typing import Dict, List, Optional, Iterable, Tuple
from difflib import SequenceMatcher
from collections import Counter
import regex

import unicodedata
# import package to read xlsx files
# import openpyxl
# import rispy
# import bibtexparser
# import itertools

# Sklearn Packages
from sklearn.metrics import cohen_kappa_score

In [2]:
# Define path to the project folder
base = Path.cwd().parent
base

# 1.1 Post Screening - Sample

In [3]:
# Load Initial Screening Results of 50 samples from ASReview

df_asreview_results_50sample = pd.read_csv(base / "data/screening/asreview_results_initialscreening.csv")

print(f"Number of rows in asreview_results_50sample: {len(df_asreview_results_50sample)}")
label_counts = df_asreview_results_50sample['asreview_label'].value_counts(normalize=True) * 100
print("\nPercentage of labels in asreview_results_50sample:")
print(label_counts)

Number of rows in asreview_results_50sample: 50

Percentage of labels in asreview_results_50sample:
asreview_label
0    90.0
1    10.0
Name: proportion, dtype: float64


# 1.2 Post Screening - Full

In [4]:
# load asreview_results.csv from data folder
df_asreview_results = pd.read_csv(base / "data/screening/asreview_results.csv")
#len of df_asreview_results

print(f'Number of rows in asreview_results: {len(df_asreview_results)}')

Number of rows in asreview_results: 1680


In [5]:
# what are the titles of df_final_prescreen that doesnt exist in df_asreview_results

df_final_prescreen = pd.read_csv(base / "data/screening/df_final_prescreen.csv")

missing_in_asreview = df_final_prescreen[~df_final_prescreen['title'].isin(df_asreview_results['title'])]['title'].tolist()
print(f"Number of titles in final combined filtered dataset missing in asreview results: {len(missing_in_asreview)}")
for title in missing_in_asreview:
    print(f" - {title}")

Number of titles in final combined filtered dataset missing in asreview results: 3
 - Learning to Generate Instruction Tuning Datasets for Zero-Shot Task Adaptation
 - Measuring and identifying factors of individuals' trust in Large Language Models
 - Measuring Morality in the 2020 US Presidential Election


In [6]:
# EDA on df_asreview_results
num_included = df_asreview_results[df_asreview_results['asreview_label'] == 1].shape[0]
num_excluded = df_asreview_results[df_asreview_results['asreview_label'] == 0].shape[0]

print(f"Number of included papers according to ASReview: {num_included}")
print(f"Number of excluded papers according to ASReview: {num_excluded}")
print(f"Total number of papers in ASReview results: {len(df_asreview_results)}")

# percentage of asreview_label 1, 0
label_counts = df_asreview_results['asreview_label'].value_counts(normalize=True) * 100
print("\nASReview Label Distribution (%):")
for label, pct in label_counts.items():
    print(f"Label {label}: {pct:.2f}%")

# distribution of asreview_user_name df_asreview_results
print("\nASReview User Contribution:")
user_counts = df_asreview_results['asreview_user_name'].value_counts()
for user, count in user_counts.items():
    print(f"{user}: {count} papers reviewed")

Number of included papers according to ASReview: 107
Number of excluded papers according to ASReview: 426
Total number of papers in ASReview results: 1680

ASReview Label Distribution (%):
Label 0.0: 79.92%
Label 1.0: 20.08%

ASReview User Contribution:
Student Screener B: 260 papers reviewed
Student Screener A: 235 papers reviewed
Lead Reviewer: 38 papers reviewed


In [7]:
# Create stratified samples of 50 papers per user for Validation
def stratified_n(df: pd.DataFrame, n_total: int = 50, 
                 label_col: str = 'asreview_label', 
                 seed: int = 42) -> pd.DataFrame:
    # Return up to n_total rows stratified by label_col (approx. proportional; min 1 per present stratum)
    if df.empty:
        return df
    n_total = min(n_total, len(df))
    counts = df[label_col].value_counts()
    props = counts / counts.sum()
    quotas = props * n_total
    base = quotas.astype(int)
    remainder = n_total - base.sum()
    frac = (quotas - base).sort_values(ascending=False)
    for lbl in frac.index[:remainder]:
        base[lbl] = base.get(lbl, 0) + 1

    parts = []
    chosen_idx = []
    for lbl, g in df.groupby(label_col):
        k = min(base.get(lbl, 0), len(g))
        if k > 0:
            take = g.sample(n=k, random_state=seed)
            parts.append(take)
            chosen_idx.extend(take.index.tolist())

    out = pd.concat(parts, ignore_index=False) if parts else df.iloc[:0]
    need = n_total - len(out)
    if need > 0:
        pool = df.drop(index=chosen_idx, errors='ignore')
        if len(pool) > 0:
            extra = pool.sample(n=min(need, len(pool)), random_state=seed)
            out = pd.concat([out, extra], ignore_index=False)

    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)

# Build 30-per-user stratified samples
samples_by_user = {}
for user in df_asreview_results['asreview_user_name'].dropna().unique():
    user_df = df_asreview_results[df_asreview_results['asreview_user_name'] == user]
    samples_by_user[user] = stratified_n(user_df, n_total=50, label_col='asreview_label', seed=42)

df_asreview_sample_screenerA   = samples_by_user.get('Student Screener A', pd.DataFrame())
df_asreview_sample_screenerB = samples_by_user.get('Student Screener B', pd.DataFrame())
df_asreview_sample_screenerAandB = pd.concat([df_asreview_sample_screenerA, df_asreview_sample_screenerB],ignore_index=True)


# For each sample, print size and label distribution
for name, sdf in [('df_asreview_sample_screenerA', df_asreview_sample_screenerA),
                  ('df_asreview_sample_screenerB', df_asreview_sample_screenerB),
                  ('df_asreview_sample_screenerAandB', df_asreview_sample_screenerAandB)]:
    if not sdf.empty:
        print(f"{name} sample size: {len(sdf)}")
        print(sdf['asreview_label'].value_counts(normalize=True).mul(100).round(2))



df_asreview_sample_screenerA sample size: 50
asreview_label
0.0    92.0
1.0     8.0
Name: proportion, dtype: float64
df_asreview_sample_screenerB sample size: 50
asreview_label
0.0    76.0
1.0    24.0
Name: proportion, dtype: float64
df_asreview_sample_screenerAandB sample size: 100
asreview_label
0.0    84.0
1.0    16.0
Name: proportion, dtype: float64


In [ ]:
# Save the Validation D.F. into csv
df_asreview_sample_screenerA.drop(columns=['asreview_label']).to_csv(base / "data/screening/df_asreview_sample_screenerA.csv", index=False)
df_asreview_sample_screenerB.drop(columns=['asreview_label']).to_csv(base / "data/screening/df_asreview_sample_screenerB.csv", index=False)
df_asreview_sample_screenerAandB.to_csv(base / "data/screening/df_asreview_sample_screenerAandB.csv", index=False)

## Compare Screeners

In [9]:
# load asreview_Validation_Screening
df_asreview_validation_of_ScreenerA = pd.read_csv(base / "data/screening/asreview_Validation_Screening_-_ScreenerA.csv")
print(f'Number of rows in df_asreview_validation_of_ScreenerA: {len(df_asreview_validation_of_ScreenerA)}')

df_asreview_validation_of_ScreenerB = pd.read_csv(base / "data/screening/asreview_Validation_Screening_-_ScreenerB.csv")
print(f'Number of rows in df_asreview_validation_of_ScreenerB: {len(df_asreview_validation_of_ScreenerB)}')

df_asreview_validation_of_ScreenerAandB = pd.read_csv(base / "data/screening/asreview_Validation_Screening_-_ScreenerA_and_ScreenerB.csv")
print(f'Number of rows in df_asreview_validation_of_ScreenerAandB: {len(df_asreview_validation_of_ScreenerAandB)}')

Number of rows in df_asreview_validation_of_ScreenerA: 50
Number of rows in df_asreview_validation_of_ScreenerB: 50


Number of rows in df_asreview_validation_of_ScreenerAandB: 100


In [10]:
# Calculate Cohen's Kappa Score between ScreenerA's and ScreenerB's
df_compare_ScreenerA_and_ScreenerB = pd.concat([
    # ScreenerA's original + ScreenerB's validation
    df_asreview_sample_screenerA[['paperId', 'title', 'asreview_label']]
        .rename(columns={'asreview_label': 'asreview_label_ScreenerA'})
        .merge(
            df_asreview_validation_of_ScreenerA[['paperId', 'asreview_label']]
                .rename(columns={'asreview_label': 'asreview_label_ScreenerB'}),
            on='paperId',
            how='inner'
        ),
    # ScreenerB's original + ScreenerA's validation
    df_asreview_sample_screenerB[['paperId', 'title', 'asreview_label']]
        .rename(columns={'asreview_label': 'asreview_label_ScreenerB'})
        .merge(
            df_asreview_validation_of_ScreenerB[['paperId', 'asreview_label']]
                .rename(columns={'asreview_label': 'asreview_label_ScreenerA'}),
            on='paperId',
            how='inner'
        )
], ignore_index=True)

# take random 50 samples from df_compare_ScreenerA_and_ScreenerB
# df_compare_ScreenerA_and_ScreenerB = df_compare_ScreenerA_and_ScreenerB.sample(n=50, random_state=42).reset_index(drop=True)

kappa_score_screenerA_screenerB = cohen_kappa_score(df_compare_ScreenerA_and_ScreenerB['asreview_label_ScreenerA'],df_compare_ScreenerA_and_ScreenerB['asreview_label_ScreenerB'])
print(f"Cohen's Kappa Score between ScreenerA's and ScreenerB's validation labels: {kappa_score_screenerA_screenerB:.2f}")

Cohen's Kappa Score between ScreenerA's and ScreenerB's validation labels: 0.60


In [ ]:
# Calcualte Cohen's Kappa Score between LeadReviewer and others's
df_compare_LeadReviewer_and_ScreenerB = (df_asreview_sample_screenerAandB.query("asreview_user_name == 'Student Screener B'")[['paperId', 'title', 'asreview_label']]
    .rename(columns={'asreview_label': 'asreview_label_ScreenerB'}).merge(df_asreview_validation_of_ScreenerAandB[['title', 'asreview_label']]
    .rename(columns={'asreview_label': 'asreview_label_LeadReviewer'}), on='title', how='inner')[['paperId', 'title', 'asreview_label_ScreenerB', 'asreview_label_LeadReviewer']]
)
print(f'Number of rows in df_compare_LeadReviewer_and_ScreenerB: {len(df_compare_LeadReviewer_and_ScreenerB)}')

df_compare_LeadReviewer_and_ScreenerA= (df_asreview_sample_screenerAandB.query("asreview_user_name == 'Student Screener A'")[['paperId', 'title', 'asreview_label']]
    .rename(columns={'asreview_label': 'asreview_label_ScreenerA'})
    .merge(df_asreview_validation_of_ScreenerAandB[['title', 'asreview_label']]
    .rename(columns={'asreview_label': 'asreview_label_LeadReviewer'}), on='title', how='inner')[['paperId', 'title', 'asreview_label_ScreenerA', 'asreview_label_LeadReviewer']]
)
print(f'Number of rows in df_compare_LeadReviewer_and_ScreenerA: {len(df_compare_LeadReviewer_and_ScreenerA)}')
screenerB_rows = (
    df_asreview_sample_screenerAandB.query("asreview_user_name == 'Student Screener B'")[['paperId', 'title']]
    .merge(
        df_asreview_validation_of_ScreenerB[['paperId', 'asreview_label']].rename(columns={'asreview_label': 'asreview_label_ScreenerA'}),
        on='paperId',
        how='left'
    )
    .merge(
        df_asreview_validation_of_ScreenerAandB[['paperId', 'asreview_label']].rename(columns={'asreview_label': 'asreview_label_LeadReviewer'}),
        on='paperId',
        how='left'
    )
    [['paperId', 'title', 'asreview_label_ScreenerA', 'asreview_label_LeadReviewer']]
)
df_compare_LeadReviewer_and_ScreenerA = (
    pd.concat([df_compare_LeadReviewer_and_ScreenerA, screenerB_rows], ignore_index=True)
      .drop_duplicates(subset=['paperId'])
      .reset_index(drop=True)
)
print(f'Number of rows in df_compare_LeadReviewer_and_ScreenerA: {len(df_compare_LeadReviewer_and_ScreenerA)}')


Number of rows in df_compare_LeadReviewer_and_ScreenerB: 50
Number of rows in df_compare_LeadReviewer_and_ScreenerB: 50
Number of rows in df_compare_LeadReviewer_and_ScreenerA: 50
Number of rows in df_compare_LeadReviewer_and_ScreenerA: 100


In [12]:
kappa_score_leadreviewer_screenerB = cohen_kappa_score(df_compare_LeadReviewer_and_ScreenerB['asreview_label_LeadReviewer'],df_compare_LeadReviewer_and_ScreenerB['asreview_label_ScreenerB'])
kappa_score_leadreviewer_screenerA = cohen_kappa_score(df_compare_LeadReviewer_and_ScreenerA['asreview_label_LeadReviewer'],df_compare_LeadReviewer_and_ScreenerA['asreview_label_ScreenerA'])
print(f"Cohen's Kappa Score between LeadReviewer's and ScreenerB's validation labels: {kappa_score_leadreviewer_screenerB:.2f}")
print(f"Cohen's Kappa Score between LeadReviewer's and ScreenerA's validation labels: {kappa_score_leadreviewer_screenerA:.2f}")

Cohen's Kappa Score between LeadReviewer's and ScreenerB's validation labels: 0.85
Cohen's Kappa Score between LeadReviewer's and ScreenerA's validation labels: 0.65


In [13]:
# Extrac LeadReviewer's validation of ScreenerA's label
df_leadreviewer_validation_of_screenerA = df_asreview_validation_of_ScreenerAandB[df_asreview_validation_of_ScreenerAandB['paperId'].isin(df_asreview_validation_of_ScreenerA['paperId'])]
df_leadreviewer_validation_of_screenerA = df_leadreviewer_validation_of_screenerA[['title', 'asreview_label']].copy()
df_leadreviewer_validation_of_screenerA['asreview_label'] = df_leadreviewer_validation_of_screenerA['asreview_label'].map({1: 'Relevant', 0: 'Not Relevant'})
df_leadreviewer_validation_of_screenerA

In [ ]:
# Extrac LeadReviewer's validation of ScreenerB's label
df_leadreviewer_validation_of_screenerB = df_asreview_validation_of_ScreenerAandB[df_asreview_validation_of_ScreenerAandB['paperId'].isin(df_asreview_validation_of_ScreenerB['paperId'])]
df_leadreviewer_validation_of_screenerB = df_leadreviewer_validation_of_screenerB[['title', 'asreview_label']].copy()
df_leadreviewer_validation_of_screenerB['asreview_label'] = df_leadreviewer_validation_of_screenerB['asreview_label'].map({1: 'Relevant', 0: 'Not Relevant'})
df_leadreviewer_validation_of_screenerB

## Fixing Student Screener A's Labels

In [15]:
# filter df_asreview_results for asreview_user_name == Student Screener A
df_asreview_screenerA = df_asreview_results[df_asreview_results['asreview_user_name'] == 'Student Screener A'].reset_index(drop=True)
print(f'Number of papers reviewed by Student Screener A: {len(df_asreview_screenerA)}')

df_asreview_screenerA = df_asreview_screenerA[~df_asreview_screenerA['paperId'].isin(df_asreview_validation_of_ScreenerA['paperId'])].reset_index(drop=True)
print(f'Number of papers reviewed by Student Screener A after removing validation papers: {len(df_asreview_screenerA)}') 

# remove the asreview_label column
df_asreview_screenerA = df_asreview_screenerA.drop(columns=['asreview_label'])
df_asreview_screenerA.to_csv(base / "data/screening/df_asreview_screenerA.csv", index=False)

Number of papers reviewed by Student Screener A: 235
Number of papers reviewed by Student Screener A after removing validation papers: 185


In [16]:
# Get ScreenerA's validated papers by LeadReviewer
df_asreview_ScreenerA_validatedByLeadReviewer = df_asreview_validation_of_ScreenerAandB[
    df_asreview_validation_of_ScreenerAandB['paperId'].isin(df_asreview_validation_of_ScreenerA['paperId'])
].reset_index(drop=True)

## Screners Contribution

After removing screener with low agreement score.

In [17]:
# Load asreview_results_Validating_ScreenerA's_Screening from data_asreview folder
df_asreview_screenerA_Fixed = pd.read_csv(base / "data/screening/asreview_results_Validating_ScreenerA_Screening.csv")
df_asreview_screenerA_Fixed

print(f'Number of rows in df_asreview_screenerA_Fixed: {len(df_asreview_screenerA_Fixed)}')
user_counts_fixed = df_asreview_screenerA_Fixed['asreview_user_name'].value_counts()
print("\nASReview User Contribution in Fixed Data:")
for user, count in user_counts_fixed.items():
    print(f"{user}: {count} papers reviewed")

Number of rows in df_asreview_screenerA_Fixed: 185

ASReview User Contribution in Fixed Data:
Student Screener B: 129 papers reviewed
Lead Reviewer: 56 papers reviewed


# 2. Studies from Databases

- All key papers have been marked as relevant; 
- At least twice the RR_T records have been screened; 
- A minimum of 10% of the total dataset has been screened;
- No relevant records have been identified in the last 50 records 

In [18]:
# Define normalization function (normalize_title)
def normalize_title(s: str) -> str:
    # Unicode normalize
    s = unicodedata.normalize("NFKC", str(s))
    # Lowercase
    s = s.lower()
    # Remove punctuation-like characters
    s = re.sub(r"[^\w\s]", " ", s)   # keep letters, numbers, underscore, whitespace
    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

## Load in ASReview Output

In [19]:
# Load ASReview Screened Outputs

# Load in asreview_results_FINAL.csv
df_asreview_results_FINAL = pd.read_csv(base / "data/screening/asreview_results_FINAL.csv")
print(f'Number of rows in df_asreview_results_FINAL: {len(df_asreview_results_FINAL)}')

# Extract Screened Papers
df_asreview_results_FINAL_screened = df_asreview_results_FINAL[df_asreview_results_FINAL['asreview_label'].notna()].reset_index(drop=True)
print(f'Number of Screened papers: {len(df_asreview_results_FINAL_screened)}')

# Extract Relevant Papers
df_asreview_results_FINAL_relevant = df_asreview_results_FINAL[df_asreview_results_FINAL['asreview_label'] == 1].reset_index(drop=True)
df_asreview_results_FINAL_relevant['norm_title'] = df_asreview_results_FINAL_relevant['title'].apply(normalize_title)
print(f'Number of Relevant papers: {len(df_asreview_results_FINAL_relevant)}')

# Percentage of ASReview Labels (relevant vs not relevant)
label_counts_final = df_asreview_results_FINAL_screened['asreview_label'].value_counts(normalize=True) * 100
print("\nFinal ASReview Label Distribution (%):")
for label, pct in label_counts_final.items():
    print(f"Label {label}: {pct:.2f}%")


Number of rows in df_asreview_results_FINAL: 1680


Number of Screened papers: 536
Number of Relevant papers: 109

Final ASReview Label Distribution (%):
Label 0.0: 79.66%
Label 1.0: 20.34%


## Validate Coverage

Compare identified papers w with the list of gold-standard papers - papers that should be in the corpus.

In [ ]:
# Load in Gold Standard Papers
df_groundtruth_papers = pd.read_csv(base / "data/search/LLM - Survey Proxies (key papers).csv")
df_groundtruth_papers = df_groundtruth_papers[["Title", "Item Type", "Abstract Note", "Publication Year", "Author", "DOI", "ISBN", "ISSN", "Date Added"]].drop_duplicates().reset_index(drop=True)
df_groundtruth_papers["preprint_flag"] = df_groundtruth_papers["Item Type"].apply(lambda x: "preprint" if x == "preprint" else "non-preprint")
df_groundtruth_papers = df_groundtruth_papers.rename(columns={"DOI": "doi", "ISBN": "isbn", "ISSN": "issn"})
df_groundtruth_papers["norm_title"] = df_groundtruth_papers["Title"].apply(normalize_title)
print(f"Gold list size: {len(df_groundtruth_papers)}")

# Remove specific papers from groundtruth list
papers_to_remove = [
    "The Potential and Challenges of Evaluating Attitudes, Opinions, and Values in Large Language Models",
    "Algorithmic Fidelity of Large Language Models in Generating Synthetic German Public Opinions: A Case Study",
    "Knowledge of cultural moral norms in large language models"
]
papers_to_remove = [normalize_title(title) for title in papers_to_remove]
df_groundtruth_papers = df_groundtruth_papers[~df_groundtruth_papers['norm_title'].isin(papers_to_remove)].reset_index(drop=True)
print(f"Gold list size: {len(df_groundtruth_papers)} (after removal of 3 papers)")

Gold list size: 26
Gold list size: 23 (after removal of 3 papers)


In [21]:
# Check the Percentage of Ground Truth Papers Matched in Screened ASReview Results
matched_by_doi = df_groundtruth_papers['doi'].isin(df_asreview_results_FINAL_screened['doi'])
num_matched_by_doi = matched_by_doi.sum()
print(f'Number of papers matched by DOI: {num_matched_by_doi} out of {len(df_groundtruth_papers)} ({(num_matched_by_doi / len(df_groundtruth_papers)) * 100:.2f}%)')

# Check by preprint vs non-preprint
df_groundtruth_papers['matched_in_asreview'] = df_groundtruth_papers['doi'].isin(df_asreview_results_FINAL_screened['doi'])
preprint_stats = df_groundtruth_papers.groupby('preprint_flag')['matched_in_asreview'].agg(['sum', 'count'])
preprint_stats['percentage_matched'] = (preprint_stats['sum'] / preprint_stats['count']) * 100
preprint_stats = preprint_stats.rename(columns={'sum': 'num_matched', 'count': 'total_papers'})
print(preprint_stats)

Number of papers matched by DOI: 17 out of 23 (73.91%)
               num_matched  total_papers  percentage_matched
preprint_flag                                               
non-preprint            12            12          100.000000
preprint                 5            11           45.454545


## Clean & Screen

- Supplement with metadata from Zotero
- Filter out Preprints
- Filter out Non-English papers

### Supplement Metadata

In [22]:
# Supplement Metedata Merge

# Load in Metedata (Supplement from Zotero)
df_supplement_metadata = pd.read_csv(base / "data/screening/asreview_results_FINAL_ZoteroSupplements.csv")
df_supplement_metadata['norm_title'] = df_supplement_metadata['Title'].apply(normalize_title)

# Supplement Metadata Merge
df_asreview_results_supplemented = df_asreview_results_FINAL_relevant.rename(columns={
    'title': 'Title',
    'doi': 'DOI',
    'year': 'Publication Year',
    'authors': 'Author'
}).merge(
    df_supplement_metadata[['norm_title', 'Item Type', 'Journal Abbreviation', 'Publisher']],
    on='norm_title',
    how='left'
)

# Fill missing values in 'Item Type', 'Journal Abbreviation', and 'Publisher' columns using DOI-based metadata
df_asreview_results_supplemented = df_asreview_results_supplemented.merge(
    df_supplement_metadata[['DOI', 'Item Type', 'Journal Abbreviation', 'Publisher']].rename(columns={
        'Item Type': 'Item Type_doi',
        'Journal Abbreviation': 'Journal Abbreviation_doi',
        'Publisher': 'Publisher_doi'
    }),
    on='DOI',
    how='left'
)
for col in ['Item Type', 'Journal Abbreviation', 'Publisher']: # Coalesce the columns to fill missing values
    df_asreview_results_supplemented[col] = df_asreview_results_supplemented[col].fillna(df_asreview_results_supplemented[f'{col}_doi'])

df_asreview_results_supplemented = df_asreview_results_supplemented.drop(columns=['Item Type_doi', 'Journal Abbreviation_doi', 'Publisher_doi']) # Drop helper columns

### Check for Status Updates

Some preprints may have changed status since the initial search. For example, some may now have been published as journal articles or presented as conference proceedings. We need to update the preprint status field accordingly.

In [23]:
# Update Preprint Status

# Import records of papers has updated preprint status
df_recordstofix_preprintstatus = pd.read_csv(base / "data/screening/recordstofix_preprintstatus.csv")

# Update record's status (preprint to conferencePaper)
df_asreview_results_supplemented_updated = df_asreview_results_supplemented.copy()
df_asreview_results_supplemented_updated.loc[df_asreview_results_supplemented_updated['norm_title'].isin(df_recordstofix_preprintstatus['norm_title']), 'Item Type'] = 'conferencePaper'

# Update record's status (journalArticle to preprint)
df_asreview_results_supplemented_updated.loc[df_asreview_results_supplemented_updated['DOI'] == '10.2139/ssrn.4548937', 'Item Type'] = 'preprint'
df_asreview_results_supplemented_updated.loc[df_asreview_results_supplemented_updated['DOI'] == '10.5281/zenodo.17077752', 'Item Type'] = 'preprint'

### Screening: Exclude Preprints

In [24]:
# Exclude Preprints
df_studies_Database = df_asreview_results_supplemented_updated.copy()
df_studies_Database = df_studies_Database[df_studies_Database['Item Type'] != 'preprint'].reset_index(drop=True)

print(f'Number of rows in df_studies_Database before excluding preprints: {len(df_asreview_results_supplemented_updated)}')
print(f'Number of Preprints in df_studies_Database: {len(df_asreview_results_supplemented_updated[df_asreview_results_supplemented_updated["Item Type"] == "preprint"])}')
print(f'Number of rows in df_studies_Database after excluding preprints: {len(df_studies_Database)}')

Number of rows in df_studies_Database before excluding preprints: 109
Number of Preprints in df_studies_Database: 48
Number of rows in df_studies_Database after excluding preprints: 61


# 3. Studies from Other Methods

In [25]:
# Load in Zotero Papers
df_studies_OtherPapers = pd.read_csv(base / "data/search/Identified papers other methods (Zotero).csv")
df_studies_OtherPapers = df_studies_OtherPapers[["Title", "Item Type", "Abstract Note", "Publication Year", "Author", "DOI", "ISBN", "ISSN", "Date Added"]].drop_duplicates().reset_index(drop=True)
df_studies_OtherPapers["preprint_flag"] = df_studies_OtherPapers["Item Type"].apply(lambda x: "preprint" if x == "preprint" else "non-preprint")
df_studies_OtherPapers = df_studies_OtherPapers.rename(columns={"DOI": "doi", "ISBN": "isbn", "ISSN": "issn"})
df_studies_OtherPapers["norm_title"] = df_studies_OtherPapers["Title"].apply(normalize_title)

print(f"List of Papers collected from 'Other methods': {len(df_studies_OtherPapers)}")
print(f"Number of conference papers: {len(df_studies_OtherPapers[df_studies_OtherPapers['Item Type'] == 'conferencePaper'])}")
print(f"Number of journal articles: {len(df_studies_OtherPapers[df_studies_OtherPapers['Item Type'] == 'journalArticle'])}")
print(f"Number of preprint papers: {len(df_studies_OtherPapers[df_studies_OtherPapers['Item Type'] == 'preprint'])}")

List of Papers collected from 'Other methods': 46
Number of conference papers: 12
Number of journal articles: 16
Number of preprint papers: 18


In [26]:
# Remove Records: Preprints & Duplicates

# Remove Preprints from df_studies_OtherPapers
df_studies_OtherPapers_filtered = df_studies_OtherPapers[df_studies_OtherPapers['preprint_flag'] != 'preprint'].reset_index(drop=True)
print(f"Number of papers after removing Preprints: {len(df_studies_OtherPapers_filtered)}")

# Remove Duplicates with Database (based on Title and DOI)
df_studies_OtherPapers_filtered = df_studies_OtherPapers_filtered[~df_studies_OtherPapers_filtered['norm_title'].isin(df_asreview_results_FINAL_relevant['norm_title'])]
df_studies_OtherPapers_filtered = df_studies_OtherPapers_filtered[~df_studies_OtherPapers_filtered['doi'].isin(df_asreview_results_FINAL_relevant['doi'])]#
print(f"Number of papers after removing duplicates with Database (based on Title and DOI): {len(df_studies_OtherPapers_filtered)}")

Number of papers after removing Preprints: 28
Number of papers after removing duplicates with Database (based on Title and DOI): 7


# 4. Merge Identified Records

In [27]:
# Combine sources into a single dataframe
df_studies_Combinedsource = pd.concat([df_studies_Database, df_studies_OtherPapers_filtered], ignore_index=True)
df_studies_Combinedsource = df_studies_Combinedsource[['Title', 'Publication Year', 'Author', 'DOI', 'Item Type', 'norm_title']]

In [ ]:
# Load in DataExtraction_Final.csv
df_DataExtraction_Final = pd.read_csv(base / "data/extraction/DataExtraction_Final.csv")

df_DataExtraction_Final["norm_title"] = df_DataExtraction_Final["Title"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
df_studies_Combinedsource["norm_title"] = df_studies_Combinedsource["Title"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()